# Mini-batch sampling for PINN training

In this notebook, we compare five mini-batch construction strategies on a benign 1D diffusion PINN, isolating the effect of *how* per-step samples are drawn from a fixed candidate pool. The
five strategies progressively layer the levers identified in the chapter:

1. **Uniform** — i.i.d. draw from the union pool (subset proportions follow pool proportions).
2. **Stratified** — fixed loss-component fractions (1/3, 1/3, 1/3) at every step.
3. **Spatial-stratified** — loss-component stratified, plus spatial-cell stratification within $\mathcal{D}_\Omega$ (8×8 = 64 cells).
4. **Causal** — loss-component stratified, plus interior $t$-coordinate restricted to $[0, T_{\rm cur}(k)]$.


We perform two experiments:
- **Experiment A** — mini-batch sampling comparison at fixed $|\mathcal{I}_k|=16$.
- **Experiment B** — batch-size sweep showing where spatial stratification helps.

We solve a 1D **advection-dominated advection-diffusion** problem:
$$u_t + c\,u_x - \mu\,u_{xx} = s(x,t), \qquad (x,t) \in [-1,1]\times[0,1],$$
with $c = 1$, $\mu = 0.01$ (Péclet number $\mathrm{Pe} = cL/\mu = 200$).
The manufactured solution is a Gaussian pulse advecting from $x=-0.5$ to $x=+0.5$:
$$u_\star(x,t) = \exp\!\bigl(-50\,(x + 0.5 - c\,t)^2\bigr),$$
which produces a residual that is sharply concentrated at the moving pulse.
Boundary and initial conditions are non-homogeneous (taken from $u_\star$).

The PINN is a small tanh-MLP (4 hidden layers, width 32, 3,297 parameters). 
We use equal-weights for three-term loss $L = \overline{r^2}_\Omega + \overline{(u-u_0)^2}_{\rm IC} +
\overline{u^2}_{\rm BC}$, so the population
minimizer matches $u_\star$.

In [ ]:
import copy, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Fix seeds so the candidate pool, network init, and training trajectories are reproducible.
torch.manual_seed(0)
np.random.seed(0)
# Use float64 throughout: PDE residuals involve second derivatives where float32 noise
# can dominate the gradient signal for small batches.
torch.set_default_dtype(torch.float64)

PI = math.pi
C_ADV = 1.0   # advection speed c
MU    = 0.01  # diffusion coefficient (Pe = c*L/mu = 200, advection-dominated)

## 1. Problem setup

Manufactured Gaussian pulse $u_\star(x,t) = \exp(-50(x+0.5-ct)^2)$ advects
across the domain over $t \in [0,1]$, staying $\sim 0.5$ away from the
boundaries. Source term cancels the advection part, leaving only
$s = -\mu\,u_{xx}$.

In [ ]:
def u_exact(xt):
    """Manufactured exact solution: Gaussian pulse advecting at speed c."""
    x, t = xt[..., 0:1], xt[..., 1:2]
    arg = x + 0.5 - C_ADV * t   # travelling-wave coordinate
    return torch.exp(-50.0 * arg ** 2)

def src(xt):
    """Source term so that u_* exactly satisfies u_t + c u_x - mu u_xx = src."""
    x, t = xt[..., 0:1], xt[..., 1:2]
    arg = x + 0.5 - C_ADV * t
    u = torch.exp(-50.0 * arg ** 2)
    # u_t + c u_x = 0 by construction (u depends only on x - c t).
    # u_xx = (10000 arg^2 - 100) * u, hence -mu u_xx = -mu (10000 arg^2 - 100) u.
    return -MU * (10000.0 * arg ** 2 - 100.0) * u

# Pool sizes: large interior pool (PDE residual) vs small IC/BC sets (only 1D each).
M_PDE = 4096
M_IC  = 80
M_BC  = 80

def make_pool(seed=1, m_pde=M_PDE):
    """Build the fixed candidate pool: i.i.d. uniform interior, equispaced IC/BC."""
    g = torch.Generator().manual_seed(seed)
    # Interior pool: uniform on [-1,1] x [0,1].
    xt_pde = torch.empty(m_pde, 2)
    xt_pde[:, 0] = torch.rand(m_pde, generator=g) * 2.0 - 1.0
    xt_pde[:, 1] = torch.rand(m_pde, generator=g)
    # IC pool: t = 0, x equispaced on [-1, 1].
    xt_ic = torch.zeros(M_IC, 2)
    xt_ic[:, 0] = torch.linspace(-1, 1, M_IC)
    # BC pool: half on x = -1, half on x = +1, t equispaced on [0, 1].
    half = M_BC // 2
    xt_bc = torch.zeros(M_BC, 2)
    xt_bc[:half, 0] = -1.0; xt_bc[:half, 1] = torch.linspace(0, 1, half)
    xt_bc[half:, 0] =  1.0; xt_bc[half:, 1] = torch.linspace(0, 1, M_BC - half)
    return xt_pde, xt_ic, xt_bc

# Dense test grid used to compute the relative L2 error during training.
nx, nt = 101, 101
xx = torch.linspace(-1, 1, nx); tt = torch.linspace(0, 1, nt)
X_TEST, T_TEST = torch.meshgrid(xx, tt, indexing='xy')
TEST_XT = torch.stack([X_TEST.reshape(-1), T_TEST.reshape(-1)], dim=1)
TEST_U  = u_exact(TEST_XT)

xt_pde_pool, xt_ic_pool, xt_bc_pool = make_pool(seed=1)

# Visualize the manufactured solution (left) and the candidate pool (right).
fig, axes = plt.subplots(1, 2, figsize=(10, 3.0))
im = axes[0].pcolormesh(X_TEST.numpy(), T_TEST.numpy(),
                        TEST_U.reshape(nt, nx).numpy(), shading='auto', cmap='viridis')
axes[0].set_xlabel('x'); axes[0].set_ylabel('t')
axes[0].set_title(r'Exact solution $u_\star(x,t) = e^{-50(x + 0.5 - ct)^2}$ (Pe=200)')
fig.colorbar(im, ax=axes[0])
axes[1].scatter(xt_pde_pool[:, 0], xt_pde_pool[:, 1], s=2, color='#003F5C', label=f'interior ({M_PDE})')
axes[1].scatter(xt_ic_pool[:,  0], xt_ic_pool[:,  1], s=8, color='#FF6E54', label=f'IC ({M_IC})')
axes[1].scatter(xt_bc_pool[:,  0], xt_bc_pool[:,  1], s=8, color='#955196', label=f'BC ({M_BC})')
axes[1].set_xlabel('x'); axes[1].set_ylabel('t'); axes[1].set_title('Candidate pool (i.i.d.)')
axes[1].legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

## 2. PINN model and loss

A small tanh-MLP (4 hidden layers, width 32, 3,297 parameters). Equal-weight
three-term loss $L = \overline{r^2}_\Omega + \overline{(u-u_0)^2}_{\rm IC} +
\overline{u^2}_{\rm BC}$ (each term a per-stratum mean), so the population
minimizer matches $u_\star$.
We utilize: 
- `residual` that computes $u_t + c\,u_x - \mu\,u_{xx} - s$ (advection-diffusion).
- IC and BC losses use the exact $u_\star$ as target (non-homogeneous).

In [ ]:
class PINN(nn.Module):
    """Small fully-connected tanh-MLP used as the trial function u_theta(x,t)."""
    def __init__(self, depth=4, width=32):
        super().__init__()
        # depth = number of hidden layers; each followed by tanh, plus a linear output head.
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, xt):
        return self.net(xt)

def residual(model, xt):
    """PDE residual r = u_t + c u_x - mu u_xx - src, computed via autograd through the network."""
    # Need create_graph=True so that backprop through the loss can differentiate the residual itself.
    g = xt.clone().detach().requires_grad_(True)
    u = model(g)
    grads = torch.autograd.grad(u, g, torch.ones_like(u), create_graph=True)[0]
    ux, ut = grads[..., 0:1], grads[..., 1:2]
    # Second derivative w.r.t. x only (column 0 of the Hessian-vector product).
    uxx = torch.autograd.grad(ux, g, torch.ones_like(ux), create_graph=True)[0][..., 0:1]
    return ut + C_ADV * ux - MU * uxx - src(g)

def batch_loss(model, b_pde, b_ic, b_bc):
    """Three-term equal-weight (1, 1, 1) PINN loss with non-homogeneous IC/BC.

    Each term is a per-stratum mean, so the loss is unbiased under stratified sampling.
    Empty mini-strata are skipped to avoid NaNs from mean over zero elements.
    """
    L = 0.0
    if len(b_pde):
        L = L + (residual(model, b_pde) ** 2).mean()
    if len(b_ic):
        u_ic_pred = model(b_ic).squeeze(-1)
        u_ic_true = u_exact(b_ic).squeeze(-1)
        L = L + ((u_ic_pred - u_ic_true) ** 2).mean()
    if len(b_bc):
        u_bc_pred = model(b_bc).squeeze(-1)
        u_bc_true = u_exact(b_bc).squeeze(-1)
        L = L + ((u_bc_pred - u_bc_true) ** 2).mean()
    return L

def l2_rel(model):
    """Relative L^2 error against the manufactured solution on the dense test grid."""
    with torch.no_grad():
        u_pred = model(TEST_XT)
    return float(torch.sqrt(((u_pred - TEST_U) ** 2).sum() / (TEST_U ** 2).sum()))

def full_pool_loss(model, xt_pde, xt_ic, xt_bc):
    """Evaluate L on the entire pool (used as the noise-free reference at step 0)."""
    with torch.no_grad():
        L_ic = ((model(xt_ic).squeeze(-1) - u_exact(xt_ic).squeeze(-1)) ** 2).mean()
        L_bc = ((model(xt_bc).squeeze(-1) - u_exact(xt_bc).squeeze(-1)) ** 2).mean()
    # PDE term needs grad enabled for the residual derivatives even though we won't backprop.
    L_pde = (residual(model, xt_pde) ** 2).mean()
    return float((L_pde + L_ic + L_bc).detach())

## 3. Mini-batch strategies

Each selector returns `(b_pde, b_ic, b_bc)` for a given total batch size `B`.

- **POOL_FRAC** = pool proportions used by `uniform_batch`, giving a *biased* gradient estimator that over-weights the (dominant) interior loss term.
- **STRAT_FRAC** = (1/3, 1/3, 1/3) — matches the equal loss weights, giving an
  *unbiased* gradient estimator.

Spatial stratification partitions $\Omega$ into $K = K_x\times K_t = 8\times 8 = 64$
equal-volume cells; the interior portion of each mini-batch is drawn so as to
cover distinct cells (one per cell when batch ≤ K, round-robin filling otherwise).

In [ ]:
# Pool proportions: biased estimator, matches the relative pool sizes (interior dominates).
POOL_FRAC  = (M_PDE/(M_PDE + M_IC + M_BC),
              M_IC /(M_PDE + M_IC + M_BC),
              M_BC /(M_PDE + M_IC + M_BC))
# Equal loss-component fractions: unbiased estimator of the equally-weighted loss.
STRAT_FRAC = (1/3, 1/3, 1/3)

def _split(B, frac):
    """Split batch budget B into (pde, ic, bc) counts; BC absorbs rounding so total = B."""
    bs_pde = round(B * frac[0])
    bs_ic  = round(B * frac[1])
    bs_bc  = B - bs_pde - bs_ic
    return bs_pde, bs_ic, bs_bc

# Spatial stratification grid: K = 8 x 8 = 64 equal-volume cells covering Omega = [-1,1]x[0,1].
SPATIAL_KX, SPATIAL_KT = 8, 8
SPATIAL_K  = SPATIAL_KX * SPATIAL_KT

def _cell_id(xt, kx=SPATIAL_KX, kt=SPATIAL_KT):
    """Map each (x,t) point to a flat cell index in [0, kx*kt)."""
    x, t = xt[:, 0], xt[:, 1]
    ix = torch.clamp(((x + 1) / 2 * kx).long(), 0, kx - 1)
    it = torch.clamp((t * kt).long(), 0, kt - 1)
    return ix * kt + it

class SpatialStratifier:
    """Precomputes cell -> pool-index lists for one-pass spatially-stratified sampling."""
    def __init__(self, xt_pool):
        self.xt = xt_pool
        cells = _cell_id(xt_pool)
        # Indices of pool points falling into each cell (cached once per pool).
        self.cell_idx = [(cells == c).nonzero(as_tuple=True)[0] for c in range(SPATIAL_K)]
        self.t_pool = xt_pool[:, 1]
    def sample(self, n_take, t_max=None):
        """Draw n_take pool indices with spatial coverage. If t_max is given, restrict to t <= t_max (causal)."""
        if t_max is None:
            cell_idx = self.cell_idx
        else:
            # Causal mode: keep only the portion of each cell with t <= t_max.
            mask = (self.t_pool <= t_max)
            cell_idx = [ci[mask[ci]] for ci in self.cell_idx]
        occupied = [c for c in range(SPATIAL_K) if len(cell_idx[c]) > 0]
        if not occupied:
            return torch.tensor([], dtype=torch.long)
        chosen = []
        if n_take <= len(occupied):
            # Small batch: one point from each of n_take distinct cells (maximum spread).
            pick = torch.randperm(len(occupied))[:n_take]
            for j in pick:
                c = occupied[j.item()]
                ci = cell_idx[c]
                chosen.append(ci[torch.randint(len(ci), (1,))].item())
        else:
            # Large batch: round-robin fill -- per points per cell, plus 1 extra to the first `rem` cells.
            per = n_take // len(occupied)
            rem = n_take - per * len(occupied)
            for jj, c in enumerate(occupied):
                n = per + (1 if jj < rem else 0)
                ci = cell_idx[c]
                if n <= len(ci):
                    pk = ci[torch.randperm(len(ci))[:n]]
                else:
                    # Fewer pool points than slots in this cell: sample with replacement.
                    pk = ci[torch.randint(len(ci), (n,))]
                chosen.extend(pk.tolist())
        return torch.tensor(chosen, dtype=torch.long)

# Cache one stratifier per pool tensor so the cell partitioning is computed only once.
_STRATIFIER_CACHE = {}
def stratifier_for(xt_pool):
    k = id(xt_pool)
    if k not in _STRATIFIER_CACHE:
        _STRATIFIER_CACHE[k] = SpatialStratifier(xt_pool)
    return _STRATIFIER_CACHE[k]

# Causal time-marching schedule: linearly grows the admissible time horizon from
# T_CUR_MIN at k=0 to 1 once k >= T_CUR_WARM * K (the "warmup" fraction of training).
T_CUR_MIN, T_CUR_WARM = 0.05, 0.3
def t_cur(k, K, t_min=T_CUR_MIN, t_warm=T_CUR_WARM):
    return t_min + (1.0 - t_min) * min(1.0, k / max(1, t_warm * K))

def uniform_batch(xt_pde, xt_ic, xt_bc, B, k=None, K=None):
    """Strategy 1: i.i.d. uniform draw with subset sizes proportional to pool sizes (biased)."""
    bs_pde, bs_ic, bs_bc = _split(B, POOL_FRAC)
    return (xt_pde[torch.randperm(len(xt_pde))[:bs_pde]],
            xt_ic[torch.randperm(len(xt_ic))[:bs_ic]],
            xt_bc[torch.randperm(len(xt_bc))[:bs_bc]])

def stratified_batch(xt_pde, xt_ic, xt_bc, B, k=None, K=None):
    """Strategy 2: i.i.d. within each loss component, but with (1/3, 1/3, 1/3) split (unbiased)."""
    bs_pde, bs_ic, bs_bc = _split(B, STRAT_FRAC)
    return (xt_pde[torch.randperm(len(xt_pde))[:bs_pde]],
            xt_ic[torch.randperm(len(xt_ic))[:bs_ic]],
            xt_bc[torch.randperm(len(xt_bc))[:bs_bc]])

def spatial_batch(xt_pde, xt_ic, xt_bc, B, k=None, K=None):
    """Strategy 3: loss-stratified + spatial-cell stratification on the interior batch."""
    bs_pde, bs_ic, bs_bc = _split(B, STRAT_FRAC)
    idx_pde = stratifier_for(xt_pde).sample(bs_pde)
    return (xt_pde[idx_pde],
            xt_ic[torch.randperm(len(xt_ic))[:bs_ic]],
            xt_bc[torch.randperm(len(xt_bc))[:bs_bc]])

def causal_batch(xt_pde, xt_ic, xt_bc, B, k, K):
    """Strategy 4: loss-stratified + restrict interior/BC samples to t <= T_cur(k) (time-marching)."""
    bs_pde, bs_ic, bs_bc = _split(B, STRAT_FRAC)
    Tk = t_cur(k, K)
    mask_pde = (xt_pde[:, 1] <= Tk).nonzero(as_tuple=True)[0]
    mask_bc  = (xt_bc[:, 1] <= Tk).nonzero(as_tuple=True)[0]
    # If the admissible region has fewer points than the batch slot, sample with replacement.
    if len(mask_pde) < bs_pde:
        idx_pde = mask_pde[torch.randint(len(mask_pde), (bs_pde,))]
    else:
        idx_pde = mask_pde[torch.randperm(len(mask_pde))[:bs_pde]]
    idx_ic = torch.randperm(len(xt_ic))[:bs_ic]
    if len(mask_bc) < bs_bc:
        idx_bc = mask_bc[torch.randint(len(mask_bc), (bs_bc,))]
    else:
        idx_bc = mask_bc[torch.randperm(len(mask_bc))[:bs_bc]]
    return xt_pde[idx_pde], xt_ic[idx_ic], xt_bc[idx_bc]

def all_batch(xt_pde, xt_ic, xt_bc, B, k, K):
    """Strategy 5: loss-stratified + spatial cells + causal time horizon (all levers combined)."""
    bs_pde, bs_ic, bs_bc = _split(B, STRAT_FRAC)
    Tk = t_cur(k, K)
    idx_pde = stratifier_for(xt_pde).sample(bs_pde, t_max=Tk)
    # Top up with uniform draws if the time-restricted spatial sampler returned too few.
    if len(idx_pde) < bs_pde:
        more = torch.randperm(len(xt_pde))[:bs_pde - len(idx_pde)]
        idx_pde = torch.cat([idx_pde, more])
    mask_bc = (xt_bc[:, 1] <= Tk).nonzero(as_tuple=True)[0]
    if len(mask_bc) < bs_bc:
        idx_bc = mask_bc[torch.randint(len(mask_bc), (bs_bc,))]
    else:
        idx_bc = mask_bc[torch.randperm(len(mask_bc))[:bs_bc]]
    return xt_pde[idx_pde], xt_ic[torch.randperm(len(xt_ic))[:bs_ic]], xt_bc[idx_bc]

# Registry of strategies + their display labels and plot colors.
SELECTORS = {'uniform': uniform_batch, 'stratified': stratified_batch,
             'spatial': spatial_batch, 'causal': causal_batch, 'all': all_batch}
LABELS = {'uniform': 'Uniform', 'stratified': 'Stratified (loss components)',
          'spatial': 'Spatial-stratified', 'causal': 'Causal (loss-strat + time)',
          'all': 'All-combined (loss + spatial + time)'}
COLORS = {'uniform': '#003F5C', 'stratified': '#955196',
          'spatial': '#1F8D60', 'causal': '#FF6E54', 'all': '#A6231D'}

## 4. Training loop

Standard mini-batch Adam at fixed learning rate $\eta = 10^{-3}$. We log
$L^2_{\rm rel}$ and the per-batch training loss every 500 iterations. No
gradient diagnostics, no learning-rate scaling — the comparison isolates
mini-batch construction.

In [ ]:
# Training hyperparameters. Held fixed across all strategies so the comparison
# isolates mini-batch construction only.
ITERS    = 15000   # total Adam steps
LR       = 1e-3    # fixed learning rate (no schedule)
BATCH    = 16      # default mini-batch size used in Experiment A
LOG_EVERY = 500    # how often to record L2 error and batch loss

def train(model, xt_pde, xt_ic, xt_bc, strategy='uniform',
          iters=ITERS, lr=LR, batch_size=BATCH, log_every=LOG_EVERY, seed=0):
    """Run Adam for `iters` steps using the selected mini-batch strategy; log L2 + loss."""
    selector = SELECTORS[strategy]
    opt = optim.Adam(model.parameters(), lr=lr)
    # Initial point: noise-free reference loss + L2 at iteration 0.
    history = {'iter': [0], 'l2': [l2_rel(model)],
               'loss': [full_pool_loss(model, xt_pde, xt_ic, xt_bc)]}
    # Seed the per-strategy stream of mini-batch draws (independent of model init).
    torch.manual_seed(seed)
    for k in range(iters):
        # k, K passed for the causal / all-combined selectors that use the curriculum schedule.
        b_pde, b_ic, b_bc = selector(xt_pde, xt_ic, xt_bc, batch_size, k=k, K=iters)
        opt.zero_grad()
        L = batch_loss(model, b_pde, b_ic, b_bc)
        L.backward()
        opt.step()
        if (k + 1) % log_every == 0:
            history['iter'].append(k + 1)
            history['l2'].append(l2_rel(model))
            # Per-batch loss only (cheap); the full-pool loss is too costly to log every step.
            history['loss'].append(float(L.detach()))
    return {kk: np.array(vv) for kk, vv in history.items()}

def ewma(x, alpha=0.4):
    """Exponentially-weighted moving average; used purely to smooth noisy trajectories for plots."""
    out = np.empty_like(x, dtype=float)
    out[0] = x[0]
    for i in range(1, len(x)):
        out[i] = alpha * x[i] + (1 - alpha) * out[i-1]
    return out

## 5. Experiment A — 5-strategy comparison ($|\mathcal{I}_k|=16$)

All five strategies trained from the same random initialization for 15,000
Adam iterations on the same fixed pool. Compares the strategies head-to-head
at a tight batch budget where bias-correction trade-offs are most visible.

In [ ]:
# Shared initial model: every strategy starts from the *same* weights so any divergence
# in the training curves is attributable to the sampling strategy, not initialisation.
torch.manual_seed(0)
init_model = PINN(depth=4, width=32)
print('# parameters:', sum(p.numel() for p in init_model.parameters()))

# Shared pool (same seed) so every strategy sees the same candidate set.
xt_pde, xt_ic, xt_bc = make_pool(seed=1, m_pde=4096)

STRATS = ['uniform', 'stratified', 'causal', 'spatial', 'all']
results_A = {}
for strat in STRATS:
    print(f'  Exp A: {strat:>11s} ...', end=' ', flush=True)
    m = copy.deepcopy(init_model)   # fresh copy of the shared init weights
    t0 = time.time()
    results_A[strat] = train(m, xt_pde, xt_ic, xt_bc, strategy=strat, batch_size=16)
    print(f'{time.time()-t0:.1f}s, L2_final={results_A[strat]["l2"][-1]:.3e}, '
          f'L2_avg5={float(np.mean(results_A[strat]["l2"][-5:])):.3e}')

# --- Plot: (a) L2 test error vs iteration, (b) per-batch training loss vs iteration ---
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
ax = axes[0]
for strat in STRATS:
    h = results_A[strat]
    # Raw trace (faint) + EWMA-smoothed (solid) so trends are readable through the noise.
    ax.semilogy(h['iter'], h['l2'],        color=COLORS[strat], lw=0.6, alpha=0.3)
    ax.semilogy(h['iter'], ewma(h['l2']),  color=COLORS[strat], lw=2.0, label=LABELS[strat])
ax.set_xlabel('iteration'); ax.set_ylabel(r'relative $L^2$ test error')
ax.grid(True, which='both', alpha=0.3); ax.legend(frameon=False, fontsize=7, loc='upper right')
ax.set_title(r'(a) $L^2$ test error (raw + EWMA)')

ax = axes[1]
for strat in STRATS:
    h = results_A[strat]
    ax.semilogy(h['iter'], ewma(h['loss']), color=COLORS[strat], lw=2.0, label=LABELS[strat])
ax.set_xlabel('iteration'); ax.set_ylabel(r'training loss $L(\theta_k;I_k)$')
ax.grid(True, which='both', alpha=0.3)
ax.set_title('(b) per-batch training loss (EWMA)')
plt.tight_layout(); plt.show()

# Export raw + smoothed curves as CSV for the book chapter (TikZ/pgfplots consumes them).
import os
RES = '/Users/ak_home/Desktop/tex_files/overleaf/25_opti_chapter/68c1792693900ed5f6cf4853/results/ex12'
os.makedirs(RES, exist_ok=True)
for strat in STRATS:
    h = results_A[strat]
    np.savetxt(f'{RES}/expA_{strat}.csv',
               np.column_stack([h['iter'], h['l2'], h['loss'], ewma(h['l2']), ewma(h['loss'])]),
               header='iter,l2,loss,l2_ewma,loss_ewma',
               delimiter=',', comments='', fmt=['%d', '%.6e', '%.6e', '%.6e', '%.6e'])

## 6. Experiment B — batch-size sweep

Sweep $|\mathcal{I}_k| \in \{4, 8, 16, 32, 64, 128, 256\}$ for four
strategies (uniform, stratified, spatial, all-combined). Causal is omitted to
keep the figure readable; its behavior tracks `all-combined` minus the spatial
contribution, which Experiment A already characterizes.

The expected pattern: spatial stratification helps most at *small* batch sizes,
where each per-step sample needs to count; at large $|\mathcal{I}_k|$ uniform's
higher per-step interior coverage closes the gap.

In [ ]:
# Sweep the batch budget |I_k| across a geometric grid. Each (strategy, B) cell trains
# from the same shared init, so the sole varying axis within a row is the batch size.
BATCHES = [4, 8, 16, 32, 64, 128, 256]
STRATS_B = ['uniform', 'stratified', 'causal', 'spatial', 'all']
results_B = {}
for strat in STRATS_B:
    for B in BATCHES:
        print(f'  Exp B: {strat:>11s} B={B:>4d} ...', end=' ', flush=True)
        m = copy.deepcopy(init_model)
        t0 = time.time()
        results_B[(strat, B)] = train(m, xt_pde, xt_ic, xt_bc, strategy=strat,
                                       batch_size=B)
        print(f'{time.time()-t0:.1f}s, L2_avg5={float(np.mean(results_B[(strat, B)]["l2"][-5:])):.3e}')

# Persist every (strategy, B) trajectory; cap L2 at 10 inside the EWMA so an early
# divergence spike doesn't dominate the smoothed curve on the chapter plots.
for (strat, B), h in results_B.items():
    l2_clip = np.minimum(h['l2'], 10.0)
    np.savetxt(f'{RES}/expB_{strat}_B{B}.csv',
               np.column_stack([h['iter'], h['l2'], h['loss'], ewma(l2_clip), ewma(h['loss'])]),
               header='iter,l2,loss,l2_ewma,loss_ewma',
               delimiter=',', comments='', fmt=['%d', '%.6e', '%.6e', '%.6e', '%.6e'])

def final_l2(strat, B):
    """Final-error summary: mean of the last 5 logged L2 values to reduce log-jitter."""
    return float(np.mean(results_B[(strat, B)]['l2'][-5:]))

# Summary table (all strategies x all batches) and per-strategy "final vs B" series for plotting.
with open(f'{RES}/expB_summary.csv', 'w') as f:
    f.write('strategy,batch,L2_final\n')
    for strat in STRATS_B:
        for B in BATCHES:
            f.write(f'{strat},{B},{final_l2(strat, B):.6e}\n')

for strat in STRATS_B:
    rows = [(B, final_l2(strat, B)) for B in BATCHES]
    np.savetxt(f'{RES}/expB_finals_{strat}.csv',
               np.array(rows),
               header='batch,l2_final',
               delimiter=',', comments='', fmt=['%d', '%.6e'])

# Plot styles. Panel (a) uses *colour = batch size* and *linestyle = strategy* so the
# uniform vs spatial comparison is readable at every B. Panel (b) uses one curve per strategy.
COLORS_B = {4: '#0B6FA4', 8: '#4078A1', 16: '#1F8D60',
            32: '#D27B11', 64: '#A6231D', 128: '#9D4E9F', 256: '#000000'}
LS = {'uniform': '-', 'stratified': '-.', 'causal': '-', 'spatial': '--', 'all': ':'}
MARKERS = {'uniform': 'o', 'stratified': '^', 'causal': 'v', 'spatial': 's', 'all': 'D'}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
ax = axes[0]
# Panel (a): only uniform vs spatial trajectories — the cleanest illustration of the
# batch-size dependence of spatial stratification.
for strat in ['uniform', 'spatial']:
    for B in BATCHES:
        h = results_B[(strat, B)]
        y = np.clip(ewma(h['l2']), None, 10)
        ax.semilogy(h['iter'], y, color=COLORS_B[B], lw=1.4, ls=LS[strat])
# Dummy legend handles: separate the colour-by-B legend from the linestyle-by-strategy legend.
for B in BATCHES:
    ax.semilogy([], [], color=COLORS_B[B], lw=1.4, label=f'$|I_k|={B}$')
ax.semilogy([], [], color='black', lw=1.4, ls='-',  label='uniform')
ax.semilogy([], [], color='black', lw=1.4, ls='--', label='spatial-stratified')
ax.set_xlabel('iteration'); ax.set_ylabel(r'$L^2$ (EWMA)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(frameon=False, fontsize=6, ncol=3, loc='upper right')
ax.set_title('(a) trajectories: uniform (solid) vs spatial-stratified (dashed)')

ax = axes[1]
# Panel (b): final L2 vs B for every strategy — the headline plot of Experiment B.
for strat in STRATS_B:
    finals = [final_l2(strat, B) for B in BATCHES]
    ax.loglog(BATCHES, finals, color=COLORS[strat], ls=LS[strat],
              marker=MARKERS[strat], lw=2, ms=7, label=LABELS[strat].split(' (')[0])
ax.set_xlabel(r'batch size $|I_k|$'); ax.set_ylabel(r'final $L^2_{\rm rel}$')
ax.grid(True, which='both', alpha=0.3)
ax.legend(frameon=False, fontsize=8)
ax.set_title('(b) final $L^2$ vs batch size (mean of last 5)')
plt.tight_layout(); plt.show()

# Print the same table to stdout for a quick text summary next to the figure.
print('\nFinal L^2 (mean of last 5):')
print(f"{'B':>4s}", *(f'{LABELS[s].split(chr(32))[0]:>11s}' for s in STRATS_B))
for B in BATCHES:
    print(f'{B:>4d}', *(f'{final_l2(s, B):>11.3e}' for s in STRATS_B))

## 7. Discussion

The considered advection-dominated PDE produces a residual that is *spatially
localized* on the moving Gaussian pulse rather than smoothly distributed
across the domain. The observed effect is therefore:

- **Spatial-stratified:**: With the residual concentrated on a small subset of cells, an i.i.d. mini-batch wastes most of its budget on cells where the residual is already small. 
Spatial stratification guarantees each per-step batch covers the active region.
- **Causal time-marching:**: The pulse advects in
  $t$, so fitting small-$t$ before larger-$t$ residuals helps to improve the performance.
- **Loss-stratified:**: The IC term encodes the
  initial peak location, and starving it of gradient signal is fatal here. Hence, utilizing loss-stratified approaches restores the imbalance and improves the convergence speed and the retained accuracy. 
